In [1]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

In [2]:
tokenizer = AutoTokenizer.from_pretrained(
    "../models/distilbert_final"
)

print(type(tokenizer))

<class 'transformers.models.bert.tokenization_bert.BertTokenizer'>


In [3]:
model = AutoModelForSequenceClassification.from_pretrained(
    "../models/distilbert_final"
)

print(type(model))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

<class 'transformers.models.distilbert.modeling_distilbert.DistilBertForSequenceClassification'>


In [4]:
print(model.config.num_labels)

2


In [5]:
print(model.config.id2label)

print(model.config.label2id)

{0: 'LABEL_0', 1: 'LABEL_1'}
{'LABEL_0': 0, 'LABEL_1': 1}


In [6]:
print(
    model.classifier.weight[:5]
)

tensor([[ 0.0256,  0.0019,  0.0157,  ...,  0.0016,  0.0120,  0.0061],
        [-0.0005,  0.0229,  0.0130,  ..., -0.0264,  0.0328, -0.0045]],
       grad_fn=<SliceBackward0>)


In [7]:
text = """
This movie was amazing.
Excellent acting.
I loved every minute.
"""

inputs = tokenizer(
    text,
    return_tensors="pt"
)

with torch.no_grad():

    outputs = model(**inputs)

print(outputs.logits)

tensor([[-3.2271,  2.9733]])


In [8]:
probabilities = torch.softmax(
    outputs.logits,
    dim=1
)

print(probabilities)

tensor([[0.0020, 0.9980]])


In [9]:
text = """
Terrible movie.
Very boring.
Waste of time.
"""

inputs = tokenizer(
    text,
    return_tensors="pt"
)

with torch.no_grad():

    outputs = model(**inputs)

print(outputs.logits)

print(
    torch.softmax(
        outputs.logits,
        dim=1
    )
)

tensor([[ 3.2191, -3.1445]])
tensor([[0.9983, 0.0017]])


In [10]:
import pandas as pd

df = pd.read_csv(
    "../data/processed/imdb_preprocessed.csv"
)

print(df.shape)

(49582, 7)


In [11]:
df["label"] = df["sentiment"].map({
    "negative": 0,
    "positive": 1
})

print(
    df["sentiment"].value_counts()
)

print()

print(
    df["label"].value_counts()
)

sentiment
positive    24884
negative    24698
Name: count, dtype: int64

label
1    24884
0    24698
Name: count, dtype: int64


In [12]:
from sklearn.metrics import accuracy_score

reviews = [
    "This movie was amazing. Excellent acting.",
    "Terrible movie. Waste of time."
]

for review in reviews:

    inputs = tokenizer(
        review,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = model(**inputs)

    pred = torch.argmax(
        outputs.logits,
        dim=1
    ).item()

    print(review)
    print("Prediction:", pred)
    print()

This movie was amazing. Excellent acting.
Prediction: 1

Terrible movie. Waste of time.
Prediction: 0



In [13]:
import pandas as pd

df = pd.read_csv(
    "../data/processed/imdb_preprocessed.csv"
)

print(df.columns)

Index(['review', 'sentiment', 'clean_review', 'tokens', 'tokens_no_stopwords',
       'tokens_lemmatized', 'final_text'],
      dtype='str')


In [14]:
print(df.head())

                                              review sentiment  \
0  One of the other reviewers has mentioned that ...  positive   
1  A wonderful little production. <br /><br />The...  positive   
2  I thought this was a wonderful way to spend ti...  positive   
3  Basically there's a family where a little boy ...  negative   
4  Petter Mattei's "Love in the Time of Money" is...  positive   

                                        clean_review  \
0  one of the other reviewers has mentioned that ...   
1  a wonderful little production the filming tech...   
2  i thought this was a wonderful way to spend ti...   
3  basically theres a family where a little boy j...   
4  petter matteis love in the time of money is a ...   

                                              tokens  \
0  ['one', 'of', 'the', 'other', 'reviewers', 'ha...   
1  ['a', 'wonderful', 'little', 'production', 'th...   
2  ['i', 'thought', 'this', 'was', 'a', 'wonderfu...   
3  ['basically', 'theres', 'a', 'family', 

In [15]:
df["label"] = df["sentiment"].map({
    "negative": 0,
    "positive": 1
})

print(df["label"].value_counts())

label
1    24884
0    24698
Name: count, dtype: int64


In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["clean_review"],
    df["label"],
    test_size=0.20,
    random_state=42,
    stratify=df["label"]
)

print(X_test.iloc[0])
print()

print("Actual Label:", y_test.iloc[0])

my wife and i watched this after dvring it off of encore action this past week it has to be the worst horror flick either of us had ever seen predictable dialogue  my wife and i were guessing the lines before they were spoken hokey special effects a screenplay that drifted all over the place i think the part that was the most annoying was the stereotyping of the various characters in the plot not to mention the gratuitous sex scene between two of the young heroines in the movie neither of which had any real purpose other than to bare certain parts of their anatomy for the cameras this movie should be categorized as comedy not horror as the villains of the movie spiders were stop motion animated and not believable in the least i cant say that i would have done a better job making a film myself but it was very amateurish and wasnt even a b movie somewhere closer to a d movie or f if that is possible i think even science fiction  would have to pass on this one

Actual Label: 0


In [17]:
import torch

def predict_sentiment(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )

    with torch.no_grad():

        outputs = model(**inputs)

    prediction = torch.argmax(
        outputs.logits,
        dim=1
    ).item()

    return prediction

In [18]:
review = X_test.iloc[0]

prediction = predict_sentiment(review)

print("Prediction:", prediction)

print("Actual:", y_test.iloc[0])

Prediction: 0
Actual: 0


In [19]:
from tqdm import tqdm

y_pred = []

for review in tqdm(X_test):

    pred = predict_sentiment(review)

    y_pred.append(pred)

100%|██████████| 9917/9917 [11:49<00:00, 13.97it/s]  


In [20]:
from sklearn.metrics import accuracy_score

print(
    accuracy_score(
        y_test,
        y_pred
    )
)

0.9145911061813048


In [21]:
print(len(X_test))
print(len(y_test))

9917
9917


In [22]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

           0       0.91      0.92      0.91      4940
           1       0.92      0.91      0.91      4977

    accuracy                           0.91      9917
   macro avg       0.91      0.91      0.91      9917
weighted avg       0.91      0.91      0.91      9917



In [23]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test,
    y_pred
)

print(cm)

[[4546  394]
 [ 453 4524]]


In [24]:
print(type(X_train))
print(type(y_train))

print(type(X_test))
print(type(y_test))

<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>


In [25]:
print(X_train.iloc[0])
print()

print("Label:", y_train.iloc[0])

i am one of jehovahs witnesses and i also work in an acute care medical facility over the years i have seen people die from hemolytic reactions to blood transfusions have attended numerous conferences on blood born pathogens and have seen several patients become seriously ill from pathogens induced by transfused blood i have also heard several jehovahs witnesses being told that they will die if they refuse blood and after  years in the field i have never actually seen it happen leaving the question is it really unreasonable to refuse blood transfusions or is the community at large benefiting from the battle on this issue the issue for jehovahs witnesses is a moral one you must abstain from blood is not an ambiguous statement thank you for this movie and allowing comments on it

Label: 1
